In [1]:
from google.colab import drive
drive.mount('/content/drive')
data_folder = "/content/drive/MyDrive/data_taxi_big/"
import polars as pl

file_list = [
    f"{data_folder}yellow_tripdata_2026-01.parquet",
    f"{data_folder}yellow_tripdata_2026-02.parquet",
    f"{data_folder}yellow_tripdata_2026-03.parquet",
    f"{data_folder}yellow_tripdata_2026-04.parquet"
]

lazy_df = pl.scan_parquet(file_list)
lazy_df.collect_schema()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Schema([('VendorID', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('RatecodeID', Int64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64),
        ('cbd_congestion_fee', Float64)])

In [2]:
total_rows_raw = lazy_df.select(pl.len()).collect().item()
print(total_rows_raw)

null_analysis = lazy_df.select(pl.all().is_null().sum()).collect()
print(null_analysis)

raw_df_sample = lazy_df.collect()
print(raw_df_sample.describe())
print(raw_df_sample.sort('trip_distance', descending=True).head(5))

14908446
shape: (1, 20)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ VendorID ┆ tpep_pick ┆ tpep_drop ┆ passenger ┆ … ┆ total_amo ┆ congestio ┆ Airport_f ┆ cbd_conge │
│ ---      ┆ up_dateti ┆ off_datet ┆ _count    ┆   ┆ unt       ┆ n_surchar ┆ ee        ┆ stion_fee │
│ u32      ┆ me        ┆ ime       ┆ ---       ┆   ┆ ---       ┆ ge        ┆ ---       ┆ ---       │
│          ┆ ---       ┆ ---       ┆ u32       ┆   ┆ u32       ┆ ---       ┆ u32       ┆ u32       │
│          ┆ u32       ┆ u32       ┆           ┆   ┆           ┆ u32       ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 0         ┆ 0         ┆ 3856909   ┆ … ┆ 0         ┆ 3856909   ┆ 3856909   ┆ 0         │
└──────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘
shape: (9, 21)
┌───────────┬───────────┬───────────┬───────────┬───

In [3]:
clean = (
    lazy_df.drop_nulls()
    .filter(
        (pl.col('passenger_count') > 0) & (pl.col('passenger_count') < 10) &
        (pl.col('trip_distance') > 0) & (pl.col('trip_distance') < 150) &
        (pl.col('fare_amount') > 2.5) & (pl.col('fare_amount') < 500) &
        (pl.col('tip_amount') >= 0) & (pl.col('tip_amount') < 200)
    )
    .filter(
        (pl.col("tpep_pickup_datetime") >= pl.datetime(2026, 1, 1)) &
        (pl.col("tpep_pickup_datetime") <= pl.datetime(2026, 4, 30, 23, 59, 59))
    )
    .with_columns(
        ((pl.col("tip_amount") / pl.col("fare_amount")) * 100).round(2).alias("tip_percentage")
    )
    .filter(pl.col("tip_percentage") <= 100)
)

In [4]:
clean = clean.with_columns([
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
    pl.col("tpep_pickup_datetime").dt.weekday().alias("pickup_day_of_week"),
    pl.col("tpep_pickup_datetime").dt.weekday().is_in([6, 7]).cast(pl.Int8).alias("is_weekend")
])

In [5]:
features_to_keep = [
    'VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
    'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount',
    'tip_percentage', 'pickup_hour', 'pickup_day_of_week', 'is_weekend'
]

final_lazy = clean.select(features_to_keep)
clean_df_real = final_lazy.collect()

print(clean_df_real.shape)
print(clean_df_real.head(5))
print(clean_df_real.describe())

(10757252, 11)
shape: (5, 11)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ VendorID ┆ tpep_pick ┆ tpep_drop ┆ passenger ┆ … ┆ tip_perce ┆ pickup_ho ┆ pickup_da ┆ is_weeken │
│ ---      ┆ up_dateti ┆ off_datet ┆ _count    ┆   ┆ ntage     ┆ ur        ┆ y_of_week ┆ d         │
│ i32      ┆ me        ┆ ime       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│          ┆ ---       ┆ ---       ┆ i64       ┆   ┆ f64       ┆ i8        ┆ i8        ┆ i8        │
│          ┆ datetime[ ┆ datetime[ ┆           ┆   ┆           ┆           ┆           ┆           │
│          ┆ μs]       ┆ μs]       ┆           ┆   ┆           ┆           ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2        ┆ 2026-01-0 ┆ 2026-01-0 ┆ 1         ┆ … ┆ 50.83     ┆ 0         ┆ 4         ┆ 0         │
│          ┆ 1         ┆ 1         ┆           ┆   ┆         

In [6]:
summary_df = clean_df_real.group_by('VendorID').agg([
    pl.col('tip_amount').mean().alias('average_tip_amount'),
    pl.col('tip_percentage').mean().alias('average_tip_percentage'),
    pl.col('trip_distance').sum().alias('total_trip_distance'),
    pl.col('passenger_count').mean().alias('average_passenger_count'),
    pl.col('VendorID').count().alias('Number_Of_trips')
])

print(summary_df)

shape: (3, 6)
┌──────────┬─────────────────┬─────────────────┬─────────────────┬────────────────┬────────────────┐
│ VendorID ┆ average_tip_amo ┆ average_tip_per ┆ total_trip_dist ┆ average_passen ┆ Number_Of_trip │
│ ---      ┆ unt             ┆ centage         ┆ ance            ┆ ger_count      ┆ s              │
│ i32      ┆ ---             ┆ ---             ┆ ---             ┆ ---            ┆ ---            │
│          ┆ f64             ┆ f64             ┆ f64             ┆ f64            ┆ u32            │
╞══════════╪═════════════════╪═════════════════╪═════════════════╪════════════════╪════════════════╡
│ 7        ┆ 3.640096        ┆ 24.369804       ┆ 470406.05       ┆ 1.213069       ┆ 179449         │
│ 1        ┆ 2.881731        ┆ 18.871459       ┆ 9513055.5       ┆ 1.099197       ┆ 2311127        │
│ 2        ┆ 3.816163        ┆ 22.196481       ┆ 2.6888e7        ┆ 1.285869       ┆ 8266676        │
└──────────┴─────────────────┴─────────────────┴─────────────────┴───────────

In [7]:
hourly_analysis = clean_df_real.group_by('pickup_hour').agg([
    pl.col('VendorID').count().alias('number_of_trips'),
    pl.col('fare_amount').mean().round(2).alias('average_fare'),
    pl.col('tip_percentage').mean().round(2).alias('average_tip_percentage')
]).sort('pickup_hour')

print("Hourly Analysis (When do people ride and tip the most?):")
print(hourly_analysis)

weekend_analysis = clean_df_real.group_by('is_weekend').agg([
    pl.col('VendorID').count().alias('number_of_trips'),
    pl.col('trip_distance').mean().round(2).alias('average_distance'),
    pl.col('tip_amount').mean().round(2).alias('average_tip')
])

print("\nWeekend vs Weekday Analysis:")
print(weekend_analysis)

Hourly Analysis (When do people ride and tip the most?):
shape: (24, 4)
┌─────────────┬─────────────────┬──────────────┬────────────────────────┐
│ pickup_hour ┆ number_of_trips ┆ average_fare ┆ average_tip_percentage │
│ ---         ┆ ---             ┆ ---          ┆ ---                    │
│ i8          ┆ u32             ┆ f64          ┆ f64                    │
╞═════════════╪═════════════════╪══════════════╪════════════════════════╡
│ 0           ┆ 255972          ┆ 20.52        ┆ 21.51                  │
│ 1           ┆ 163834          ┆ 18.29        ┆ 21.43                  │
│ 2           ┆ 104309          ┆ 16.95        ┆ 21.38                  │
│ 3           ┆ 72186           ┆ 17.89        ┆ 20.59                  │
│ 4           ┆ 51841           ┆ 24.24        ┆ 18.05                  │
│ …           ┆ …               ┆ …            ┆ …                      │
│ 19          ┆ 673336          ┆ 18.38        ┆ 24.01                  │
│ 20          ┆ 620759          ┆ 18.86 

In [8]:
!pip install pyspark -q


In [10]:
import os
import tempfile
import warnings
warnings.filterwarnings('ignore')

import polars as pl
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
import plotly.graph_objects as go
features_to_keep = [
    'VendorID', 'passenger_count', 'trip_distance', 'fare_amount',
    'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'tip_amount'
]

df_ml_ready = clean_df_real.select(features_to_keep)
print(f"Ready for PySpark ML: {df_ml_ready.shape[0]:,} rows")

Ready for PySpark ML: 10,757,252 rows


In [11]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("Taxi ML Full Data") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

sdf = spark.createDataFrame(clean_df_real.to_arrow())

In [12]:
feature_cols = ['trip_distance', 'passenger_count']
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')
lr = LinearRegression(featuresCol='features', labelCol='fare_amount', maxIter=10)

ml_pipeline = Pipeline(stages=[assembler, lr])

train_df, test_df = sdf.randomSplit([0.8, 0.2], seed=42)

pipeline_model = ml_pipeline.fit(train_df)
preds = pipeline_model.transform(test_df)

In [15]:
preds.select('fare_amount', 'prediction', 'trip_distance').show(5)

evaluator_rmse = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='rmse')
evaluator_mae = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='mae')
evaluator_r2 = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction', metricName='r2')

rmse = evaluator_rmse.evaluate(preds)
mae = evaluator_mae.evaluate(preds)
r2 = evaluator_r2.evaluate(preds)

print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

+-----------+------------------+-------------+
|fare_amount|        prediction|trip_distance|
+-----------+------------------+-------------+
|       26.8| 25.12784145083817|          5.0|
|       11.4|14.338527172957619|          2.0|
|       24.7|24.048910023050112|          4.7|
|       37.5|43.829319532497784|         10.2|
|       70.0| 72.60082427351259|         18.2|
+-----------+------------------+-------------+
only showing top 5 rows
RMSE: 7.1987
MAE:  3.5305
R²:   0.8381


In [16]:
final_columns = [
    'VendorID',
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'tip_amount',
    'tip_percentage',
    'pickup_hour',
    'pickup_day_of_week',
    'is_weekend',
    'prediction'
]

preds.select(final_columns).write.mode("overwrite").parquet("taxi_descriptive_predictive_bi.parquet")
print("Success")

Success


In [20]:
import polars as pl

single_df = pl.read_parquet("taxi_descriptive_predictive_bi.parquet/*.parquet")

single_df.write_parquet("taxi_final_single_file.parquet")

print("Success: 'taxi_final_single_file.parquet' is ready for download!")

Success: 'taxi_final_single_file.parquet' is ready for download!


In [18]:
spark.stop()